# 1.1.1 SDDB消除原CSV里大量的空行

In [ ]:
import pandas as pd

SDDB = pd.read_csv('SDDB(direct_search).csv')

SDDB["Dream Text"] = SDDB["Dream Text"].str.replace(r"[\r\n]+", " ", regex = True)
SDDB["Dream Text"] = SDDB["Dream Text"].str.replace(r"\s+", " ", regex = True)

SDDB.to_csv("SDDB_Cleaned_V1", index = False)

# 1.1.2 SDDB调用LLM清洗填写质量低下的数据

In [ ]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

#  这个api现在已经被我disable了
client = OpenAI(
    api_key='sk-684f2d6b8ab749debf024f7e5565cdb9',
    base_url="https://api.deepseek.com"
    )

df = pd.read_csv("SDDB_Cleaned_V1.csv")
text_column = 'Dream Text'

df = df.dropna(subset=[text_column]).copy()
df['text_clean'] = df[text_column].astype(str).str.strip()
df = df[df['text_clean'] != '']
df['word_count'] = df['text_clean'].str.split().str.len()

# 为了免得llm得把上万条数据全过一遍：如果某一条数据的梦境文本长度大于20个词，一般认为填写者好好写了，就直接不给llm了
# 只有梦境文本长度小于等于20个词，才觉得填写者有可能是没好好写（虽然也可能是好好写了但是就是写的很短），就提交给llm
# 还是打印一下大概交给llm的有多少条
mask_whitelist = df['word_count'] > 20
mask_grey_area = (df['word_count'] > 0) & (df['word_count'] <= 20)
df_grey = df[mask_grey_area].copy()

print(f"Word Count > 20: {mask_whitelist.sum()} ")
print(f"Word Count <= 20: {len(df_grey)} ")

# 调用llm！
def llm_is_valid_dream_concurrent(item):
    idx, text = item
    prompt = f"""
    Is this text a genuine description of a dream? 
    Reply 1 if it describes a dream/event.
    Reply 0 if it is an evasion, refusal, non-response, or random noise.
    
    Text: "{text}"
    Reply ONLY with 1 or 0.
    """
        
    res = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}], 
        temperature=0, 
        max_tokens=2
    )
    is_valid = res.choices[0].message.content.strip() == '1'
    return idx, is_valid

tasks = [(idx, text) for idx, text in df_grey['text_clean'].items()]
results_dict = {}

MAX_WORKERS = 20 
    
from concurrent.futures import ThreadPoolExecutor, as_completed
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(llm_is_valid_dream_concurrent, task) for task in tasks]
        
    for future in tqdm(as_completed(futures), total=len(futures), desc="🚀 DeepSeek 洗数中"):
        idx, is_valid = future.result()
        results_dict[idx] = is_valid
            
df_grey['is_valid_llm'] = df_grey.index.map(results_dict)

valid_grey_index = df_grey[df_grey['is_valid_llm'] == True].index
df_final = df[mask_whitelist | df.index.isin(valid_grey_index)]

print("清洗完毕")

# 保存干净的数据
df_final.to_csv("SDDB.csv", index=False)

# 1.2.1 爬取DreamBank的资料

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

headers = {
    'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.7680.80'
}
session = requests.Session()
with open("dreams.txt", "w", encoding="utf-8") as f:
    f.write("Series,Dream_Text,Word_Count\n")
# 获取所有梦境 ID 
response = session.get('https://dreambank.net/search.cgi', headers=headers, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
series = []
select_series = soup.find_all('select', {'name': 'series'})
if select_series:
    for i in select_series[0].find_all("option"):
        series.append(i.get("value"))
print(f"成功获取到 {len(series)} 个系列。")
IDs = []

for index, i in enumerate(series, 1):
    d = []
    url = f"{'https://dreambank.net/search.cgi'}?series={i}"
    query = {'query': 'e'}
    response = session.get(url, headers=headers, params=query, timeout=10)
    time.sleep(1) 
    soup = BeautifulSoup(response.text, "html.parser") 
    select_d = soup.find_all('select', {'name': 'd'})
    if select_d:
        for j in select_d[0].find_all("option"):
            d.append(j.get("value"))
        IDs.append({'name': i, 'd': d})

# 获取并保存具体的梦境内容
start_marker = '<br style="margin-bottom:-0.7em;"/>'
end_marker = '<hr noshade'
for k in IDs:
    current_name = k["name"]
    current_d = k["d"]
    print(f"开始抓取系列{current_name}")
    for count, l in enumerate(current_d, 1):
        query = {"series": current_name, "d": l, "query": "e"}
        response = session.get("https://dreambank.net/show.cgi", headers=headers, params=query, timeout=10)
        html = response.text
        time.sleep(0.75)
        if start_marker in html and end_marker in html:
            temp = html.split(start_marker, 1)[1]
            dream_html = temp.split(end_marker)[0]
            raw_text = BeautifulSoup(dream_html, "html.parser").get_text(separator=" ", strip=True)
            clean_text = raw_text.replace('\n', ' ').replace('\r', ' ').strip()
            word_count = len(clean_text.split())
            safe_text = clean_text.replace('"', '""')
            with open("dreams.txt" , "a", encoding="utf-8") as f:
                f.write(f'{current_name},"{safe_text}",{word_count}\n')
                    
print("额啊啊啊啊啊爬完了爬完了爬完了爬完了爬完了爬完了爬完了爬完了爬完了爬完了爬完了爬完了")

# 这时会生成一个dream.txt

# 1.2.2 继1.2.1后你会发现kenneth没爬到，因此补爬

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

headers = {
    'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.7680.80'
}
session = requests.Session()

url = f"{'https://dreambank.net/search.cgi'}?series={'kenneth'}"
query = {'query': 'e'}
response = session.get(url, headers=headers, params=query, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
d_list = []
select_d = soup.find_all('select', {'name': 'd'})
if select_d:
    for j in select_d[0].find_all("option"):
        d_list.append(j.get("value"))

# 抓取正文
    start_marker = '<br style="margin-bottom:-0.7em;"/>'
    end_marker = '<hr noshade'
    for count, dream_id in enumerate(d_list, 1):
        query = {"series": 'kenneth', "d": dream_id, "query": "e"}
        response = session.get("https://dreambank.net/show.cgi", headers=headers, params=query, timeout=10)
        html = response.text
        time.sleep(0.75)
        if start_marker in html and end_marker in html:
            temp = html.split(start_marker, 1)[1]
            dream_html = temp.split(end_marker)[0]
            soup_text = BeautifulSoup(dream_html, "html.parser")
            clean_text = soup_text.get_text(separator="\n", strip=True)
            with open("dreams_kenneth.txt", "a", encoding="utf-8") as f:
                f.write(f"--- Series: {'kenneth'} | ID: {dream_id} ---\n")
                f.write(clean_text + "\n\n")         
        else:
            print(f"还是有inconsistent的结构！")
                
print("额啊啊啊啊啊啊修完了这一块")

# 用这时dreams_kenneth.txt中的部分直接copy paste覆盖dreams.txt的对应内容

# 1.2.3 修复DB中的德文

In [ ]:
from openai import OpenAI
import concurrent.futures

client = OpenAI(
    api_key="sk-684f2d6b8ab749debf024f7e5565cdb9", 
    base_url="https://api.deepseek.com"
)

# 德文系列名单
german_series = [
    "vonuslar.de", 
    "german-f.de", 
    "german-m.de", 
    "zurich-f.de", 
    "zurich-m.de"
]

MAX_WORKERS = 20 

def translate_german(text):
    if not text.strip():
        return text
    system_prompt = "You are a professional translator. Translate the following German text into English. Output ONLY the English translation without any extra comments."
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content.strip()

def process_dream(dream):
    if dream["is_german"]:
        final_text = translate_german(dream["text"])
        dream["text"] = final_text
        dream["translated"] = True
    else:
        dream["translated"] = False
    
    return dream

if __name__ == "__main__":
    dreams_list = []
# 把整个文本文件切分成一个一个的梦境对象
    with open("dreams.txt", "r", encoding="utf-8") as f_in:
        current_header = ""
        current_text_lines = []
        is_german = False
        
        for line in f_in:
            if line.startswith("--- Series:"):
# 保存上一个梦境
                if current_header:
                    dreams_list.append({
                        "header": current_header,
                        "text": "".join(current_text_lines).strip(),
                        "is_german": is_german
                    })      
# 初始化当前新梦境
                current_header = line
                current_text_lines = []
                is_german = any(f"Series: {g}" in line for g in german_series)
            else:
                current_text_lines.append(line)      
# 兜底文件末尾的最后一个梦境
        if current_header:
            dreams_list.append({
                "header": current_header,
                "text": "".join(current_text_lines).strip(),
                "is_german": is_german
            })

    with open("dreams_translated.txt", "w", encoding="utf-8") as f_out:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            for result in executor.map(process_dream, dreams_list):
                f_out.write(result["header"])
                f_out.write(result["text"] + "\n\n")

    print("翻译完了翻译完了翻译完了翻译完了翻译完了翻译完了翻译完了翻译完了")

# 同理，用这时dreams_translated.txt中的部分直接copy paste覆盖dreams.txt的对应内容，最后转成DB.csv即可

# 1.3.1 针对SDDB.csv构建特征

主要是针对SDDB.csv，产出：
SDDB_VADER.csv
SDDB_EMPATH.csv
SDDB_SPACY.csv

In [ ]:
# SDDB_VADER

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

data = pd.read_csv("SDDB_clean.csv")

data["Dream Text"] = data["Dream Text"].fillna("")
dream_texts = data["Dream Text"].tolist()
dream_text_scores = []
for dream in dream_texts:
    score = analyzer.polarity_scores(dream)
    dream_text_scores.append(score)

scores_df = pd.DataFrame(dream_text_scores)
SDDB_VADER = pd.concat([data, scores_df], axis=1)
SDDB_VADER.to_csv("SDDB_VADER.csv", index = False)

# 构建difference和log特征
import pandas as pd
import numpy as np

SDDB_VADER = pd.read_csv("SDDB_VADER.csv")
SDDB_VADER["difference"] = SDDB_VADER["pos"] - SDDB_VADER["neg"]

# 这里问了ai，ai说如果硬算log的话，要么会出现log(0)，要么会出现极端的数值；所以为了稳定性，在算log时加了一个数
epsilon = 1e-5

SDDB_VADER["log"] = np.log((SDDB_VADER["pos"] + epsilon) / (SDDB_VADER["neg"] + epsilon))
SDDB_VADER.to_csv("SDDB_VADER_complex.csv", index = False)

In [ ]:
# SDDB_EMPATH
%pip install empath

from empath import Empath
import pandas as pd

lexicon = Empath()

SDDB = pd.read_csv("SDDB.csv")
SDDB_theme = SDDB["Dream_Text"].apply(lambda x: lexicon.analyze(x, normalize=True))
SDDB_theme_df = pd.DataFrame(SDDB_theme.tolist())
SDDB_EMPATH = pd.concat([SDDB, SDDB_theme_df], axis = 1)
SDDB_EMPATH.to_csv("SDDB_EMPATH.csv", index = False, encoding = "utf-8")

In [ ]:
# DB_SPACY(此代码在kaggle上运行)
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

def get_luxurious_spacy_features(text):
    if pd.isna(text) or not isinstance(text, str):
        return {} 
    doc = nlp(text)
    
# 实体类
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]
    noun_chunks = [chunk.text for chunk in doc.noun_chunks]
    
# 动作与描述类 
    verbs = [token.lemma_ for token in doc if token.pos_ == "VERB"]
    adjectives = [token.lemma_ for token in doc if token.pos_ == "ADJ"] 
    
# 统计类
    sentence_count = len(list(doc.sents)) 
    word_count = len([token for token in doc if not token.is_punct]) 

    return {
        "person_list": ", ".join(persons),
        "location_list": ", ".join(locations),
        "noun_chunks": ", ".join(noun_chunks),
        "action_verbs": ", ".join(verbs),
        "adjectives": ", ".join(adjectives),
        "sentence_count": sentence_count,
        "word_count": word_count
    }

DB = pd.read_csv("/kaggle/input/datasets/ameliaye13579/python-big-homework/DB.csv")
spacy_df = pd.DataFrame(spacy_series.tolist())
DB_FINAL = pd.concat([DB, spacy_df], axis=1)
DB_FINAL.to_csv("/kaggle/working/DB_FINAL_FEATURES.csv", index=False)
print("kaggle任务完成！")

# 1.3.2 针对DB.csv构建特征

主要是针对SDDB.csv，产出：
DB_VADER.csv
DB_EMPATH.csv
DB_SPACY.csv

In [ ]:
# DB_VADER

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

data = pd.read_csv("DB_clean.csv")

data["Dream Text"] = data["Dream Text"].fillna("")
dream_texts = data["Dream Text"].tolist()
dream_text_scores = []
for dream in dream_texts:
    score = analyzer.polarity_scores(dream)
    dream_text_scores.append(score)

scores_df = pd.DataFrame(dream_text_scores)
DB_VADER = pd.concat([data, scores_df], axis=1)
DB_VADER.to_csv("DB_VADER.csv", index = False)

# 构建difference和log特征
import pandas as pd
import numpy as np

SDDB_VADER = pd.read_csv("SDDB_VADER.csv")
SDDB_VADER["difference"] = SDDB_VADER["pos"] - SDDB_VADER["neg"]

# 这里问了ai，ai说如果硬算log的话，要么会出现log(0)，要么会出现极端的数值；所以为了稳定性，在算log时加了一个数
epsilon = 1e-5

SDDB_VADER["log"] = np.log((SDDB_VADER["pos"] + epsilon) / (SDDB_VADER["neg"] + epsilon))
SDDB_VADER.to_csv("SDDB_VADER_complex.csv", index = False)

In [ ]:
# DB_EMPATH
%pip install empath

from empath import Empath
import pandas as pd

lexicon = Empath()

DB = pd.read_csv("DB.csv")
DB_theme = DB["Dream_Text"].apply(lambda x: lexicon.analyze(x, normalize=True))
DB_theme_df = pd.DataFrame(DB_theme.tolist())
DB_EMPATH = pd.concat([DB, DB_theme_df], axis = 1)
DB_EMPATH.to_csv("DB_EMPATH.csv", index = False, encoding = "utf-8")

In [ ]:
# DB_SPACY(此代码在kaggle上运行)
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

def get_luxurious_spacy_features(text):
    if pd.isna(text) or not isinstance(text, str):
        return {} 
    doc = nlp(text)
    
# 实体类
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]
    noun_chunks = [chunk.text for chunk in doc.noun_chunks]
    
# 动作与描述类 
    verbs = [token.lemma_ for token in doc if token.pos_ == "VERB"]
    adjectives = [token.lemma_ for token in doc if token.pos_ == "ADJ"] 
    
# 统计类
    sentence_count = len(list(doc.sents)) 
    word_count = len([token for token in doc if not token.is_punct]) 

    return {
        "person_list": ", ".join(persons),
        "location_list": ", ".join(locations),
        "noun_chunks": ", ".join(noun_chunks),
        "action_verbs": ", ".join(verbs),
        "adjectives": ", ".join(adjectives),
        "sentence_count": sentence_count,
        "word_count": word_count
    }

DB = pd.read_csv("/kaggle/input/datasets/ameliaye13579/python-big-homework/DB.csv")
spacy_df = pd.DataFrame(spacy_series.tolist())
DB_FINAL = pd.concat([DB, spacy_df], axis=1)
DB_FINAL.to_csv("/kaggle/working/DB_FINAL_FEATURES.csv", index=False)
print("kaggle任务完成！")